# `C01` — Order validation and rejection

Scenario **C01** of the Track A certification ladder (card `OMIE-CERT-C01-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C1). Oracle kind: **exact-parametric, reusing M4's no-mutation relation**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; every run is done both ways and the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. *Structural equality* (a byte-identical book) means two states equal leaf for leaf.

## 1. Purpose

**Purpose.** Before certifying what the book does with an order, certify which orders are allowed in.
The validator enforces §2.3's price and quantity bounds and tick grids and §4.2's iceberg peak
constraint, in fused integer arithmetic **after a tick-snap** — which produces a subtle documented
behaviour nobody has certified: a price off the grid by *less* than half a tick is snapped to the
nearest tick and **accepted**, while one off by half a tick or more commits to a different tick and
may then fall out of bounds. That boundary is exactly where a validator quietly diverges from its
specification.

There is a second, structural justification that raises this scenario's priority: **every later
scenario's random draw assumes its generated orders are accepted**. If the validator silently
rejected, say, an off-tick draw, a downstream scenario could pass *vacuously* with zero trades.
Certifying acceptance first is what makes every subsequent draw trustworthy — and it is why invariant
I9 exists.

Scope note: two of §5.4's five rejection conditions — the agent guarantee/operating limit and the TSO
unavailability check — are **not implemented**. The notebook names them as known non-coverage so the
certificate cannot read as though §5.4 were fully certified.

**Certifies** (card, with the r2 additions below): §2.3 bounds and ticks; the iceberg peak constraint $0 < \mathit{peak} \le \mathit{quantity}$ — its lower half $0 < \mathit{peak}$ **derived from §4.2, not stated by it** (a validator invariant), its upper half $\mathit{peak} \le \mathit{quantity}$ also stated by §5.4 since `market-spec.md` v2.14; §5.4's implemented subset: the price and quantity bounds, the peak-larger-than-total condition and (r2) the ICEwPI price-increment sign rule, added to §5.4 in v2.14 and implemented by `OMIE-ICEWPI-SIGN-REJECT-1`.

**Does not certify.**
- §5.4's agent guarantee / operating-limit condition and its TSO unavailability / limitation condition: **not implemented**, so not exercised (known non-coverage).
- §5.4's "after GCT" condition: that is C00's status gate (and C11's expiry), not the validator.
- *Which* §5.4 condition rejected an order: the engine does not expose a rejection reason (the battery lists `rejection_reason` as unobserved). C01 certifies accept versus reject, never the reason.
- An offset of **exactly** half a tick: it is a tie that a binary float cannot represent on a decimal grid, so it never occurs in the dataset and is not certified.
- That off-grid prices are a market behaviour at all. §2.3 allows "2 decimal places maximum"; the snap-then-check rule is the simulator's **documented behaviour** (domain review §2), and C01 certifies the simulator against that documentation (section 5, note 1).
- The **magnitude** of a price increment. §5.4 puts no rule-level bound on it; C01 submits one price tick of either sign and certifies the sign rule only. What a price increment does to later slices (the drift) is C09's.
- An iceberg past the 64-slice ceiling: that is C14e's. Every iceberg C01 expects to be accepted needs at most 64 slices (section 4), so the ceiling is never approached.

**r2 (K7-U REPEAT at K2, 2026-10-06).** Four changes from r1, all in the judging, none in the rule.
1. The `iceberg_peak_one_tick` control is bounded to at most 64 slices **by construction**, so no draw is discarded because of it and the sample is no longer biased to small icebergs.
2. The eager runs are done on the **first valid seeds** (one per side), never on seeds fixed in advance that the capacity check may discard.
3. Every assertion records its **per-draw integers in full**: expected and observed, one entry per valid seed. Nothing is `null` on a pass, and no list is truncated.
4. Two ICEwPI controls (after `OMIE-ICEWPI-SIGN-REJECT-1`): a **wrong-sign** price increment, which §5.4 (v2.14) rejects, and its right-sign twin, which is accepted, so a validator that rejected every price-incremented iceberg cannot pass.

## 2. The rule, quoted

The governing passages, quoted **verbatim** and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`.

How the quotes ground the expectation:

- **Bounds and ticks (§2.3, certified).** §2.3 is a bullet list, not a table (domain review §10). Its four bound and tick bullets are quoted one by one. The integer bounds of section 5 come from these decimal figures.
- **Inclusive bounds (§5.4, certified).** §5.4 rejects a price *below* −9,999 or *above* +9,999 and a quantity *below* 0.1 or *above* 999 MW, so the bound values themselves are accepted.
- **What a rejection means for the book (§5.1, §5.2).** A rejected order ends at `REJECTED` and never enters the SOB, so it receives no timestamp. The book, the trade tape and the engine's timestamp counter must be exactly what they were: M4's no-mutation relation, applied to every rejection.
- **The iceberg peak (§4.2 derived, §5.4 stated).** §4.2 gives an iceberg a *visible peak quantity*, the part *shown in the order book*, carved out of a *hidden total quantity*, and the data feed identifies icebergs by a *non-zero* peak. From these, $0 < \mathit{peak}$ is **derived**, not stated (domain review §10). Since v2.14, §5.4 also **states** the upper half: a peak greater than the total is rejected, and a peak equal to the total is valid. On acceptance only the peak is visible; the rest is hidden.
- **The ICEwPI sign (§5.4 v2.14 and §4.2; certified, r2).** §4.2 gives the direction of the price increment per side (a buy iceberg's slices go down, a sell iceberg's go up) and says the data feed carries it in the PPD field, 0 for a standard iceberg. §5.4 (v2.14) rejects a PPD of the wrong sign: a BUY with PPD > 0 or a SELL with PPD < 0. PPD = 0 always passes this check, and the magnitude has no rule-level bound.
- **The tick-snap (domain review §2, OQ-16).** `market-spec.md` does not say what happens to an off-grid price. The snap-then-check behaviour is quoted from the domain review, which calls it *documented behaviour*. The validator's encoding is quoted from OQ-16, whose exactness claim covers **on-tick** inputs only (section 5, notes 1 and 2).
- **Known non-coverage (§2.3 last bullet, §5.4 conditions 3 and 5).** Quoted so the certificate names them; nothing is asserted about them. §5.4's fourth condition (after GCT) is quoted for completeness; it is C00's.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- certified: §2.3 bounds and ticks -------------------------------------------------
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Price bounds:** −9,999 EUR/MWh (minimum) to +9,999 EUR/MWh (maximum).",
        note="The price bounds. In the offset encoding of section 5 they are integer ticks 0 and 1,999,800.",
    ),
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Price tick (minimum increment):** 0.01 EUR/MWh (2 decimal places maximum).",
        note="The price grid. One price tick is 0.01 EUR/MWh.",
    ),
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Quantity bounds:** 0.1 MW (minimum) to 999 MW (maximum).",
        note="The quantity bounds: integer ticks 1 and 9,990.",
    ),
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Quantity tick (minimum increment):** 0.1 MW (1 decimal place maximum).",
        note="The quantity grid. One quantity tick is 0.1 MW; it also measures an iceberg's peak.",
    ),
    # --- certified: §5.4's implemented subset ---------------------------------------------
    SpecQuote(
        "market-spec.md", "§5.4",
        "Price outside product min/max (below −9,999 or above +9,999 EUR/MWh).",
        note="§5.4, first condition. 'Below' and 'above': the bound values themselves are accepted.",
    ),
    SpecQuote(
        "market-spec.md", "§5.4",
        "Quantity outside product min/max (below 0.1 or above 999 MW).",
        note="§5.4, second condition. Inclusive bounds again.",
    ),
    # --- certified: what a rejection means ------------------------------------------------
    SpecQuote(
        "market-spec.md", "§5.1",
        "├─ REJECTED (fails OMIE or XBID validation)",
        note="A rejected order ends at REJECTED: it is never ACCEPTED, so it never rests, trades or has a fate.",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "Each order receives a timestamp when entering the SOB.",
        note="A rejected order never enters the SOB, so the engine's timestamp counter must not move.",
    ),
    # --- certified (derived): the iceberg peak --------------------------------------------
    SpecQuote(
        "market-spec.md", "§4.2",
        "A regular limit order with hidden total quantity and visible peak quantity.",
        note="The peak is part of the total: peak <= quantity (derived here; stated by §5.4 since v2.14, quoted below).",
    ),
    SpecQuote(
        "market-spec.md", "§4.2",
        "Only the peak size is shown in the order book.",
        note="On acceptance the visible quantity is the peak; the remainder is hidden.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2",
        "iceberg orders are identified by the execution restriction field = `ICE` and a non-zero "
        "\"Cantidad reducida\" (peak quantity) value.",
        note="An iceberg's peak is non-zero, and a peak that shows nothing (<= 0) is no iceberg: 0 < peak (derived).",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "§10",
        "the iceberg `0 < peak ≤ quantity` constraint is **derived** from §4.2 rather than stated by it, "
        "and is labelled a validator invariant",
        note="The domain review's own wording of the derived constraint.",
    ),
    # --- certified (stated, v2.14): the peak upper bound and the ICEwPI sign (r2) -----------
    SpecQuote(
        "market-spec.md", "§5.4",
        "**Iceberg peak larger than total.** An iceberg whose visible peak (\"Cantidad reducida\") is greater "
        "than its total quantity is rejected. A peak equal to the total is valid.",
        note="§5.4 (v2.14) now states the upper half of the peak constraint: peak <= quantity, equality accepted.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2",
        "**ICEwPI (Iceberg with Price Increment):** Each new slice is priced at the previous slice price ± a "
        "fixed delta. Buy icebergs: negative delta (price decreases per slice). Sell icebergs: positive delta "
        "(price increases per slice).",
        note="The direction of the increment per side. The first slice rests at the base price.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2",
        "The PPD field carries the per-slice price increment (0 for standard icebergs, non-zero for ICEwPI).",
        note="PPD is the increment, a price difference: n_d = PPD / price tick (section 5, note 6).",
    ),
    SpecQuote(
        "market-spec.md", "§5.4",
        "**ICEwPI price-increment sign.** An iceberg with a non-zero price increment (PPD) of the wrong sign "
        "is rejected: a BUY with PPD > 0, or a SELL with PPD < 0. PPD = 0 is a standard iceberg and is always "
        "valid on this check. The magnitude of PPD has no rule-level bound",
        note="The sign rule (r2): icewpi_wrong_sign is rejected, icewpi_right_sign accepted. Magnitude not certified.",
    ),
    # --- the tick-snap: documented simulator behaviour, not a market-spec rule -------------
    SpecQuote(
        "scenario-ladder-review.md", "§2",
        "The tick-snap boundary (sub-half-tick offsets are accepted and snapped; ≥half-tick commits to a "
        "different tick) is documented behaviour that nobody had certified.",
        note="The only written source of the off-grid expectation: market-spec.md is silent on off-grid prices.",
    ),
    SpecQuote(
        "open-questions.md", "OQ-16",
        "This produces the exact tick index for on-tick inputs at OMIE defaults "
        "(`price_tick = 0.01 EUR/MWh`, `qty_tick = 0.1 MW`).",
        block="What is known",
        note="The validator encodes a float price to an integer tick before checking bounds. The exactness "
             "claim is for on-tick inputs only; off-grid precision is not documented (section 5, note 2).",
    ),
    # --- known non-coverage: quoted, not certified ----------------------------------------
    SpecQuote(
        "market-spec.md", "§2.3",
        "The economic value of the order must not exceed the agent's operating limit (guarantee) deposited at OMIE.",
        note="NOT CERTIFIED (not implemented): the operating-limit rule.",
    ),
    SpecQuote(
        "market-spec.md", "§5.4",
        "Agent's operating limit (guarantee) would be exceeded.",
        note="NOT CERTIFIED (not implemented): §5.4, third condition.",
    ),
    SpecQuote(
        "market-spec.md", "§5.4",
        "Order fails TSO unavailability/limitation check (sum of offered energy + prior program + existing "
        "book + new order must stay within declared capacity bounds).",
        note="NOT CERTIFIED (not implemented): §5.4, fifth condition.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "§2",
        "§5.4's guarantee/operating-limit and TSO-unavailability conditions are unimplemented and named as "
        "non-coverage.",
        note="The domain review's record of the non-coverage.",
    ),
    SpecQuote(
        "market-spec.md", "§5.4",
        "Order submitted after GCT for the target contract.",
        note="NOT CERTIFIED HERE: §5.4, fourth condition, is the status gate certified by C00 (and C11).",
    ),
)
CITATIONS = ()

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"quotes: {sum(c.ok for c in QUOTE_CHECKS)}/{len(QUOTE_CHECKS)} match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`**, the card's default for a random scenario: the number of **valid** draws. One draw is one seed of the generator's `C01.validation` construction, extended in section 4 by the K2 boundary controls. Each draw submits one background order and then 27 cases. A draw that breaks a capacity margin (I8 invalid) is discarded and re-drawn with the next seed, up to `MAX_ATTEMPTS = 2 × DRAWS` seeds; the discard rate is reported. By construction (section 4) no C01 draw should be discarded.
- **`EAGER_DRAWS = 2`.** Every draw runs jitted, the production mode. Eager runs on the **first valid seeds**, decided as the run goes and never fixed in advance: a valid seed (one I8 has not discarded) runs eager while fewer than `EAGER_DRAWS` seeds have, and only if its side has no eager seed yet or both sides already have one. With 2, that is the first valid BUY seed and the first valid SELL seed, each covering every case class; they are compared field for field with their jitted run. `EAGER_SEEDS` lists the seeds actually run eager; it is empty until section 7 fills it. Eager runs every operation uncompiled, at a cost of roughly a second per few submissions, so running all 500 draws eager would exceed the card's runtime budget many times over. The eager/jitted comparison tests the compiler, not the rule.
- **`SEED0 = 0`**, the first seed tried.
- The tick grids and bounds are **not** parameters. They are read from `OMIEProductConfig()` and checked against §2.3 in section 7. The expectation (section 5) uses §2.3's decimal figures, not the config. Nor is the slice margin: `SLICE_MARGIN = 64` is the generator's margin from `scenario-ladder.md`, "The capacity-validity precondition".

In [ ]:
CERT_ID = "C01"     # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500         # valid seeds of C01.validation, all run jitted (card: random scenarios default to 500 draws)
EAGER_DRAWS = 2     # the first EAGER_DRAWS *valid* seeds also run eager (first valid BUY seed, first valid SELL seed)
SEED0 = 0           # first seed tried

In [ ]:
import pathlib
from fractions import Fraction

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side

CONSTRUCTION = "C01.validation"
CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, ICE = int(ExecRestriction.NON), int(ExecRestriction.ICE)

# §2.3's figures, as exact decimals. The expectation is built from these, never from CONFIG
# (CONFIG is checked against them in section 7).
SPEC = {
    "price_min": Fraction("-9999"), "price_max": Fraction("9999"), "price_tick": Fraction("0.01"),
    "qty_min": Fraction("0.1"), "qty_max": Fraction("999"), "qty_tick": Fraction("0.1"),
}
# The generator's slice margin (scenario-ladder.md, "The capacity-validity precondition"):
# every iceberg that enters the book needs at most ceil(total / peak) <= 64 slices.
SLICE_MARGIN = 64

SEEDS = list(range(SEED0, SEED0 + DRAWS))   # prepared up front; section 7 prepares more if seeds are discarded
MAX_ATTEMPTS = 2 * DRAWS   # the draw loop stops at DRAWS valid draws or after 2 x DRAWS attempts
EAGER_SEEDS = []           # filled by section 7 with the first valid seeds it runs eager (never fixed in advance)

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C01_order_validation.ipynb"

print(f"{CERT_ID}: {CONSTRUCTION} + K2 boundary controls; {DRAWS} valid draws from seed {SEED0} "
      f"(at most {MAX_ATTEMPTS} attempts); eager on the first {EAGER_DRAWS} valid seeds (one per side)")
print(f"OMIEProductConfig: price [{CONFIG.price_min}, {CONFIG.price_max}] tick {CONFIG.price_tick} EUR/MWh; "
      f"qty [{CONFIG.qty_min}, {CONFIG.qty_max}] tick {CONFIG.qty_tick} MW")
print(f"capacity envelope: {ENVELOPE}; slice margin {SLICE_MARGIN}")

## 4. Dataset

**The generator's cases.** `C01.validation` (`omie_tape_synth.make_validation_cases`) draws one order per validation class for each seed and carries them in `draw.params["cases"]`. Its tape is empty because the construction has no book. Each case has a price in EUR/MWh, a quantity in MW and, for an iceberg, a peak in MW. Off-grid cases also carry their drawn offset in ticks. The eleven generator cases:

| case | class (card) | how it is drawn |
|---|---|---|
| `in_bounds_on_tick` | in-bounds on-tick | price on the grid in [20, 120] EUR/MWh, quantity on the grid in [1, 50] MW |
| `price_below_min`, `price_above_max` | price below / above bound | −10,000 and +10,000 EUR/MWh |
| `qty_below_min`, `qty_above_max` | quantity below / above bound | 0.03 MW and 1,000 MW |
| `sub_half_tick_off_grid` | sub-half-tick off-grid | the on-tick price + an offset drawn in [0.1, 0.4] tick |
| `over_half_tick_off_grid` | ≥ half-tick off-grid, in bounds | the on-tick price + an offset drawn in [0.6, 0.9] tick |
| `near_max_sub_half_tick` | sub-half-tick off-grid at the bound | +9,999 + an offset drawn in [0.05, 0.45] tick |
| `near_max_over_half_tick` | ≥ half-tick off-grid, out of bounds | +9,999 + an offset drawn in [0.55, 0.95] tick |
| `iceberg_peak_exceeds_qty`, `iceberg_peak_zero` | iceberg peak > quantity / peak ≤ 0 | peak = quantity + 1 MW, and peak = 0 |

**The K2 boundary controls: sixteen more cases per seed, derived here from the same draw.** The generator has no accepted iceberg, no exact bound value, no off-grid case at the lower bound and no price-incremented iceberg. Without these controls, a validator that rejected every iceberg (or every ICEwPI), or that got the lower price bound or the inclusive bound values wrong, would pass. Each control is a fixed arithmetic function of the seed's drawn values. Below, $q_0$ and $n_{q_0}$ are the seed's on-tick quantity and its integer (in 0.1 MW ticks, so $10 \le n_{q_0} \le 500$):

| case | price | quantity | peak | PPD (BUY seed / SELL seed) |
|---|---|---|---|---|
| `price_at_min`, `price_at_max` | −9,999 and +9,999 exactly | $q_0$ | — | — |
| `price_one_tick_below_min`, `price_one_tick_above_max` | −9,999.01 and +9,999.01 | $q_0$ | — | — |
| `near_min_sub_half_tick`, `near_min_over_half_tick` | −9,999 minus the seed's two near-max offsets (mirrored) | $q_0$ | — | — |
| `qty_at_min`, `qty_at_max` | on-tick price | 0.1 and 999 MW exactly | — | — |
| `qty_zero`, `qty_one_tick_above_max` | on-tick price | 0 and 999.1 MW | — | — |
| `iceberg_peak_equals_qty` | on-tick price | $q_0$ | $= q_0$ (1 slice) | 0 |
| `iceberg_peak_one_tick` | on-tick price | $n \times 0.1$ MW, $n = 2 + (n_{q_0} \bmod 63) \in [2, 64]$ | 0.1 MW ($n$ slices, **≤ 64**) | 0 |
| `iceberg_peak_one_tick_over_qty` | on-tick price | $q_0$ | $q_0 + 0.1$ MW | 0 |
| `iceberg_peak_negative` | on-tick price | $q_0$ | −0.1 MW | 0 |
| `icewpi_right_sign` | on-tick price | $q_0$ | $= q_0$ (1 slice) | −0.01 / +0.01 EUR/MWh |
| `icewpi_wrong_sign` | on-tick price | $q_0$ | $= q_0$ (1 slice) | **+0.01 / −0.01** EUR/MWh |

**The slice margin (r2).** Every iceberg the expectation accepts needs $\lceil n_q / n_k \rceil \le 64$ slices, the generator's margin in `scenario-ladder.md` ("The capacity-validity precondition"); the engine flags `truncated` at registration only from 66 slices. In r1, `iceberg_peak_one_tick` used the full on-tick quantity ($n_{q_0}$ slices, up to 500), so most draws were discarded. It now follows the ladder's by-construction recipe for a one-tick peak: draw the slice count $n \in [2, 64]$ first and set the total to $n$ ticks. The slice count still varies with the seed. The other accepted icebergs need one slice. Section 7 re-checks the margin from every seed's cases before the run and reports the largest slice count. Rejected icebergs never enter the book, so they carry no slice count (a peak ≤ 0 has none).

**The ICEwPI controls (r2).** Both are standard on every axis except the price increment: on-tick price, the seed's quantity, a one-slice peak. The increment is one price tick, the smallest non-zero magnitude, so the only thing that differs between the two is its sign. A one-slice peak keeps a wrongly accepted wrong-sign iceberg inside every capacity margin, so its error shows as a verdict failure, not as a discarded draw.

**How each draw is submitted.** The generator does not fix this; it is the scenario's own design.
- One fresh book per seed, on one OPEN contract.
- One side for every order of the seed: **BUY on even seeds, SELL on odd seeds**. Nothing can cross, so no trade is possible.
- First a **background order**: order id 1, NON, at the seed's on-tick price and quantity. Every later submission therefore meets a non-empty book.
- Then the 27 cases, in the order of the two tables above (generator cases first), with order ids 2, 3, …, 28. Each is NON unless it is an iceberg (ICE), with validity GFS and agent 1. Every iceberg carries its PPD: 0 for a standard iceberg, the table's value for the two ICEwPI controls.
- Each case is submitted with its **unsnapped** price, quantity, peak and PPD in EUR/MWh and MW (`price_eur`, `qty_mw`, `peak_qty_mw`, `ppd_eur`). Section 6 must not snap them first, and reports what it submitted (`submitted_price_eur`, `submitted_qty_mw`, `submitted_peak_mw`, `submitted_ppd_eur`; `None` where the case has no such field).

Accepted cases rest on the seed's side. A rejected case must leave everything untouched.

**The integer tape (`TAPES[seed]`).** The same orders, written as integer ticks. Each case's price, quantity, peak and PPD are written at the tick they commit to: the nearest tick, computed in exact rational arithmetic (`nearest_tick` below). The drawn floats are taken at their exact binary value and §2.3's grids at their exact decimal value. An exact half-tick tie cannot occur, because a binary float is never an odd multiple of half a decimal tick. An integer tape cannot carry an off-grid price, so on the tape an off-grid case already sits on its committed tick. The tape therefore checks the bounds and the PPD sign on committed integers, and the float submission checks the snap itself. The tape also feeds the invariant battery.

Below: the first seed's cases in readable units beside the integers they commit to, a per-seed summary, and a plot of where every price case sits relative to the half-tick boundaries at the two price bounds.

In [ ]:
from omiecid_simulator.omiecid.encoding import decode_price, decode_qty


def nearest_tick(x, tick, origin=Fraction(0)):
    """Nearest integer to (x - origin) / tick, in exact rational arithmetic.

    `x` is taken at its exact value (a float's exact binary value). Returns
    (n, distance from the half-tick tie in ticks); an exact tie returns (None, 0).
    """
    r = (Fraction(x) - origin) / tick
    n = r.numerator // r.denominator            # floor
    frac = r - n
    if frac == Fraction(1, 2):
        return None, Fraction(0)
    return (n + 1 if frac > Fraction(1, 2) else n), abs(frac - Fraction(1, 2))


def price_ticks(p):
    """The price's committed integer in the offset encoding: nint((p - price_min) / price_tick)."""
    return nearest_tick(p, SPEC["price_tick"], SPEC["price_min"])


def qty_ticks(q):
    """The quantity's (or peak's) committed integer: nint(q / qty_tick)."""
    return nearest_tick(q, SPEC["qty_tick"])


def ppd_ticks(d):
    """The price increment's committed integer: nint(ppd / price_tick) (signed, no offset)."""
    return nearest_tick(d, SPEC["price_tick"])


def dec(n, tick):
    """The float nearest to the exact decimal n * tick."""
    return float(Fraction(n) * tick)


def k2_cases(gen_cases, side):
    """The sixteen K2 boundary controls, from the seed's generator cases and side."""
    by = {c["name"]: c for c in gen_cases}
    p_ok, q_ok = float(by["in_bounds_on_tick"]["price_eur"]), float(by["in_bounds_on_tick"]["qty_mw"])
    n_q0 = qty_ticks(q_ok)[0]
    near_sub = float(by["near_max_sub_half_tick"]["offset_ticks"])
    near_over = float(by["near_max_over_half_tick"]["offset_ticks"])
    pmin, pmax, pt = float(SPEC["price_min"]), float(SPEC["price_max"]), float(SPEC["price_tick"])
    qmin, qmax, qt = float(SPEC["qty_min"]), float(SPEC["qty_max"]), float(SPEC["qty_tick"])
    # one-tick peak: slice count n in [2, 64] drawn first (from the seed's quantity), total = n ticks
    n_one = 2 + n_q0 % (SLICE_MARGIN - 1)
    q_one = dec(n_one, SPEC["qty_tick"])
    # the ICEwPI sign: BUY slices go down (PPD < 0), SELL slices go up (PPD > 0); one price tick
    right = -pt if side == BUY else pt
    return [
        {"name": "price_at_min", "price_eur": pmin, "qty_mw": q_ok},
        {"name": "price_at_max", "price_eur": pmax, "qty_mw": q_ok},
        {"name": "price_one_tick_below_min", "price_eur": pmin - pt, "qty_mw": q_ok},
        {"name": "price_one_tick_above_max", "price_eur": pmax + pt, "qty_mw": q_ok},
        {"name": "near_min_sub_half_tick", "price_eur": pmin - near_sub * pt, "qty_mw": q_ok,
         "offset_ticks": -near_sub},
        {"name": "near_min_over_half_tick", "price_eur": pmin - near_over * pt, "qty_mw": q_ok,
         "offset_ticks": -near_over},
        {"name": "qty_at_min", "price_eur": p_ok, "qty_mw": qmin},
        {"name": "qty_at_max", "price_eur": p_ok, "qty_mw": qmax},
        {"name": "qty_zero", "price_eur": p_ok, "qty_mw": 0.0},
        {"name": "qty_one_tick_above_max", "price_eur": p_ok, "qty_mw": qmax + qt},
        {"name": "iceberg_peak_equals_qty", "price_eur": p_ok, "qty_mw": q_ok, "peak_qty_mw": q_ok,
         "exec_restriction": ICE, "ppd_eur": 0.0},
        {"name": "iceberg_peak_one_tick", "price_eur": p_ok, "qty_mw": q_one, "peak_qty_mw": qt,
         "exec_restriction": ICE, "ppd_eur": 0.0},
        {"name": "iceberg_peak_one_tick_over_qty", "price_eur": p_ok, "qty_mw": q_ok,
         "peak_qty_mw": q_ok + qt, "exec_restriction": ICE, "ppd_eur": 0.0},
        {"name": "iceberg_peak_negative", "price_eur": p_ok, "qty_mw": q_ok, "peak_qty_mw": -qt,
         "exec_restriction": ICE, "ppd_eur": 0.0},
        {"name": "icewpi_right_sign", "price_eur": p_ok, "qty_mw": q_ok, "peak_qty_mw": q_ok,
         "exec_restriction": ICE, "ppd_eur": right},
        {"name": "icewpi_wrong_sign", "price_eur": p_ok, "qty_mw": q_ok, "peak_qty_mw": q_ok,
         "exec_restriction": ICE, "ppd_eur": -right},
    ]


def build_cases(draw):
    """Every case of one draw, in submission order, with its side, order id and committed integers."""
    side = BUY if draw.seed % 2 == 0 else SELL
    gen = [dict(c, source="generator") for c in draw.params["cases"]]
    k2 = [dict(c, source="K2 control") for c in k2_cases(draw.params["cases"], side)]
    out = []
    for i, c in enumerate(gen + k2):
        ex = int(c.get("exec_restriction", NON))
        ice = ex == ICE
        n_p, gap_p = price_ticks(c["price_eur"])
        n_q, gap_q = qty_ticks(c["qty_mw"])
        n_k, gap_k = qty_ticks(c["peak_qty_mw"]) if ice else (0, None)
        ppd = float(c.get("ppd_eur", 0.0)) if ice else None
        n_d, gap_d = ppd_ticks(ppd) if ice else (0, None)
        out.append({
            "name": c["name"], "source": c["source"], "side": side, "oid": 2 + i,
            "exec_restriction": ex, "price_eur": float(c["price_eur"]), "qty_mw": float(c["qty_mw"]),
            "peak_qty_mw": float(c["peak_qty_mw"]) if ice else None,
            "ppd_eur": ppd,
            "offset_ticks": None if c.get("offset_ticks") is None else float(c["offset_ticks"]),
            "price_int": n_p, "qty_int": n_q, "peak_int": n_k if ice else 0, "ppd_int": n_d if ice else 0,
            # distance of the exact values from a half-tick tie, in ticks
            "tie_gap": min(float(g) for g in (gap_p, gap_q, gap_k, gap_d) if g is not None),
        })
    return side, out


def background(draw, side):
    """Order id 1: NON at the seed's on-tick price and quantity, on the seed's side."""
    c = next(c for c in draw.params["cases"] if c["name"] == "in_bounds_on_tick")
    return {"name": "background", "side": side, "oid": 1, "exec_restriction": NON,
            "price_eur": float(c["price_eur"]), "qty_mw": float(c["qty_mw"]),
            "price_int": price_ticks(c["price_eur"])[0], "qty_int": qty_ticks(c["qty_mw"])[0]}


def build_tape(bg, cases):
    """The integer tape: background, then every case at its committed integers (written verbatim)."""
    tb = synth.TapeBuilder(CONFIG)
    tb.add(side=bg["side"], price_int=bg["price_int"], qty_int=bg["qty_int"], order_id=bg["oid"])
    for c in cases:
        ice = c["exec_restriction"] == ICE
        tb.add(side=c["side"], price_int=c["price_int"], qty_int=c["qty_int"], order_id=c["oid"],
               exec_restriction=c["exec_restriction"],
               peak_int=c["peak_int"] if ice else None, ppd_int=c["ppd_int"] if ice else None)
    return tb.build()


DRAWN, SIDES, CASES, BACKGROUND, TAPES = {}, {}, {}, {}, {}


def prepare(seed):
    """Draw seed `seed` and derive its cases, background order and integer tape."""
    d = synth.CONSTRUCTIONS[CONSTRUCTION](seed)
    side, cases = build_cases(d)
    bg = background(d, side)
    DRAWN[seed], SIDES[seed], CASES[seed], BACKGROUND[seed] = d, side, cases, bg
    TAPES[seed] = build_tape(bg, cases)
    return d


for s in SEEDS:
    prepare(s)

rows = ["| oid | case | source | side | ex | price (EUR/MWh, as drawn) | offset (ticks) | price_int | "
        "price_int (EUR/MWh) | qty (MW) | qty_int | peak (MW) | peak_int | PPD (EUR/MWh) | ppd_int |",
        "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
_bg = dict(BACKGROUND[SEED0], source="background", offset_ticks=None, peak_qty_mw=None, peak_int=0,
           ppd_eur=None, ppd_int=0)
for c in [_bg] + CASES[SEED0]:
    ice = c["exec_restriction"] == ICE
    rows.append(
        f"| {c['oid']} | `{c['name']}` | {c['source']} | {'BUY' if c['side'] == BUY else 'SELL'} | "
        f"{'ICE' if ice else 'NON'} | {c['price_eur']!r} | "
        f"{'' if c['offset_ticks'] is None else format(c['offset_ticks'], '+.4f')} | {c['price_int']} | "
        f"{decode_price(c['price_int'], CONFIG):.2f} | {c['qty_mw']!r} | {c['qty_int']} | "
        f"{'' if c['peak_qty_mw'] is None else repr(c['peak_qty_mw'])} | "
        f"{c['peak_int'] if ice else ''} | {'' if c['ppd_eur'] is None else repr(c['ppd_eur'])} | "
        f"{c['ppd_int'] if ice else ''} |")
display(Markdown(f"**Seed {SEED0}** (side {'BUY' if SIDES[SEED0] == BUY else 'SELL'}; generator digest "
                 f"`{DRAWN[SEED0].digest()[:16]}…`; integer tape {TAPES[SEED0].shape}, "
                 f"{int(np.sum(TAPES[SEED0][:, synth.COL_OID] >= 1))} real rows)\n\n" + "\n".join(rows)))

for s in SEEDS[:5]:
    by = {c["name"]: c for c in CASES[s]}
    print(f"seed {s} ({'BUY' if SIDES[s] == BUY else 'SELL'}): on-tick {by['in_bounds_on_tick']['price_eur']:.2f} "
          f"EUR/MWh x {by['in_bounds_on_tick']['qty_mw']:.1f} MW; offsets sub "
          f"{by['sub_half_tick_off_grid']['offset_ticks']:.3f}, over {by['over_half_tick_off_grid']['offset_ticks']:.3f}, "
          f"near-max sub {by['near_max_sub_half_tick']['offset_ticks']:.3f}, near-max over "
          f"{by['near_max_over_half_tick']['offset_ticks']:.3f}; one-tick iceberg "
          f"{by['iceberg_peak_one_tick']['qty_int']} slices; ppd_int right {by['icewpi_right_sign']['ppd_int']:+d}, "
          f"wrong {by['icewpi_wrong_sign']['ppd_int']:+d}")
print(f"... {len(SEEDS)} seeds prepared, {sum(len(v) for v in CASES.values())} cases, "
      f"{sum(1 for s in SEEDS if SIDES[s] == BUY)} BUY seeds / {sum(1 for s in SEEDS if SIDES[s] == SELL)} SELL seeds")
TIE_GAP_MIN = min(c["tie_gap"] for v in CASES.values() for c in v)
print(f"smallest distance of any case from a half-tick tie: {TIE_GAP_MIN:.4f} tick")

In [ ]:
import matplotlib.pyplot as plt

# Where the off-grid and bound price cases sit, in ticks outside the nearest price bound
# (+ = outside), over every seed. The dashed lines are the half-tick boundaries.
NEAR = ("near_max_sub_half_tick", "near_max_over_half_tick", "price_at_max", "price_one_tick_above_max",
        "near_min_sub_half_tick", "near_min_over_half_tick", "price_at_min", "price_one_tick_below_min")
_top = (SPEC["price_max"] - SPEC["price_min"]) / SPEC["price_tick"]
fig, ax = plt.subplots(figsize=(7, 3.2))
for y, name in enumerate(NEAR):
    xs = []
    for s in SEEDS:
        c = next(c for c in CASES[s] if c["name"] == name)
        r = (Fraction(c["price_eur"]) - SPEC["price_min"]) / SPEC["price_tick"]
        xs.append(float(r - _top) if "max" in name else float(-r))
    ax.scatter(xs, [y] * len(xs), s=6, alpha=0.5)
ax.axvline(0, color="black", lw=0.8)
for x in (-0.5, 0.5):
    ax.axvline(x, color="grey", lw=0.8, ls="--")
ax.set_yticks(range(len(NEAR)))
ax.set_yticklabels(NEAR, fontsize=7)
ax.set_xlabel("ticks outside the nearest price bound (exact)")
ax.set_title(f"{CONSTRUCTION} + K2 controls: price cases at the bounds, {len(SEEDS)} seeds")
plt.tight_layout()
plt.show()


## 5. Known result

**Notation.** All arithmetic is exact (rational numbers). $\tau_p = 0.01$ EUR/MWh and $\tau_q = 0.1$ MW are §2.3's ticks; $p_{\min} = -9{,}999$ and $p_{\max} = +9{,}999$ EUR/MWh; $q_{\min} = 0.1$ and $q_{\max} = 999$ MW. $\operatorname{nint}(x)$ is the nearest integer to $x$. An exact half cannot occur (section 4), and section 4 prints the smallest distance of any case from one. For a case with price $p$, quantity $q$ and, for an iceberg, peak $k$ and price increment $d$ (PPD), the integers the order **commits to** (the snap) are:

$$n_p = \operatorname{nint}\!\Big(\frac{p - p_{\min}}{\tau_p}\Big), \qquad n_q = \operatorname{nint}\!\Big(\frac{q}{\tau_q}\Big), \qquad n_k = \operatorname{nint}\!\Big(\frac{k}{\tau_q}\Big), \qquad n_d = \operatorname{nint}\!\Big(\frac{d}{\tau_p}\Big).$$

In these units the bounds are $N_P = (p_{\max} - p_{\min})/\tau_p = 1{,}999{,}800$, $N_{q,\min} = 1$ and $N_{q,\max} = 9{,}990$.

**(V) The verdict** (§2.3, §5.4; the lower peak bound derived from §4.2; the PPD sign from §5.4, v2.14):

$$\text{accept} \iff 0 \le n_p \le N_P \;\wedge\; N_{q,\min} \le n_q \le N_{q,\max} \;\wedge\; \big(\text{ICE} \Rightarrow 0 < n_k \le n_q \;\wedge\; S(\text{side}, n_d)\big),$$

$$S(\text{BUY}, n_d) \iff n_d \le 0, \qquad S(\text{SELL}, n_d) \iff n_d \ge 0.$$

The bounds are inclusive (§5.4: *below* / *above*). The snap comes first: a price within half a tick of a grid point commits to that tick, and only then is it checked against the bounds. $n_d = 0$ (a standard iceberg) satisfies $S$ on either side; a NON order carries no PPD.

**(A) On accept.** The case's submission writes no trade row. The book changes by exactly one resting entry, on the seed's side, with the case's order id, at price $n_p$ (an ICEwPI's first slice rests at its base price: the increment applies only to later slices, §4.2). Its visible quantity is $n_q$ for NON, or $n_k$ for ICE (only the peak is shown, §4.2). Its hidden quantity is $n_q - \text{visible}$: 0 for NON and for an iceberg whose peak equals its quantity. Every other resting entry on that side, the whole other side and the trade tape are unchanged, and no capacity flag is raised.

**(R) On reject.** This is M4's no-mutation relation, applied to every rejection (§5.1, §5.2). The engine reports not accepted, and the submission writes no trade row. The state after the submission is **byte-identical** to the state before it, leaf for leaf: `bids`, `asks`, `trades`, `iceberg_meta`, `n_active`, `ts_counter`, `book_full` and `truncated`. `ts_counter` is included because a rejected order never enters the SOB, so it receives no timestamp.

**(T) On the integer tape.** The same verdict, on the committed integers. An accepted case's order id is among the record's accepted orders. A rejected case's is among its rejections, with reason `validator`. The background order (order id 1) is accepted and rests on both paths, and nothing trades.

**What each class gives, by construction.** $n_{p_0}$ is the integer of the seed's on-tick price and $n_{q_0}$ that of its quantity; $\delta$ is the drawn offset in ticks; $n = 2 + (n_{q_0} \bmod 63)$.

| case | $n_p$ | verdict |
|---|---|---|
| `in_bounds_on_tick` | $n_{p_0}$ | accept |
| `price_below_min`, `price_above_max` | $-100$, $N_P + 100 = 1{,}999{,}900$ | reject |
| `qty_below_min`, `qty_above_max` | $n_{p_0}$ ($n_q = 0$, $10{,}000$) | reject |
| `sub_half_tick_off_grid` ($0.1 \le \delta \le 0.4$) | $n_{p_0}$ (**snapped**) | accept |
| `over_half_tick_off_grid` ($0.6 \le \delta \le 0.9$) | $n_{p_0} + 1$ | accept |
| `near_max_sub_half_tick` ($0.05 \le \delta \le 0.45$) | $N_P$ (**snapped onto the bound**) | accept |
| `near_max_over_half_tick` ($0.55 \le \delta \le 0.95$) | $N_P + 1$ | reject |
| `iceberg_peak_exceeds_qty`, `iceberg_peak_zero` | $n_{p_0}$ ($n_k = n_q + 10$, $0$) | reject |
| `price_at_min`, `price_at_max` | $0$, $N_P$ | accept |
| `price_one_tick_below_min`, `price_one_tick_above_max` | $-1$, $N_P + 1$ | reject |
| `near_min_sub_half_tick` (mirrored $\delta$) | $0$ (**snapped onto the bound**) | accept |
| `near_min_over_half_tick` (mirrored $\delta$) | $-1$ | reject |
| `qty_at_min`, `qty_at_max` | $n_{p_0}$ ($n_q = 1$, $9{,}990$) | accept |
| `qty_zero`, `qty_one_tick_above_max` | $n_{p_0}$ ($n_q = 0$, $9{,}991$) | reject |
| `iceberg_peak_equals_qty` | $n_{p_0}$ ($n_k = n_q = n_{q_0}$, 1 slice, $n_d = 0$) | accept |
| `iceberg_peak_one_tick` | $n_{p_0}$ ($n_q = n \in [2, 64]$, $n_k = 1$, $n$ slices, $n_d = 0$) | accept |
| `iceberg_peak_one_tick_over_qty`, `iceberg_peak_negative` | $n_{p_0}$ ($n_k = n_q + 1$, $-1$) | reject |
| `icewpi_right_sign` | $n_{p_0}$ ($n_k = n_q$, $n_d = -1$ BUY / $+1$ SELL) | accept |
| `icewpi_wrong_sign` | $n_{p_0}$ ($n_k = n_q$, $n_d = +1$ BUY / $-1$ SELL) | reject |

Every accepted iceberg needs $\lceil n_q / n_k \rceil \le 64$ slices (section 4), so no accepted case can raise `truncated`, and (A)'s "no capacity flag is raised" is a real assertion, not a ceiling artefact.

The code below computes every value from the case's own drawn price, quantity, peak and PPD: `draw.params`, and the K2 controls derived from it. It never reads the generator's `draw.expected`. It also cross-checks the class table above (section 7 records the cross-check). Every asserted value is an integer, a boolean or an order id; no euro figure is asserted.

**Under-specification notes (K2).**
1. *Off-grid prices are not in `market-spec.md`.* §2.3 says "2 decimal places maximum", which in the real market reads as "a price with more decimals is invalid". The snap-then-check rule (snap and accept below half a tick) is the simulator's **documented behaviour**, quoted from the domain review (§2); it is not a market rule. The off-grid rows above certify the simulator against that documentation.
2. *Precision of the snap.* OQ-16 records the validator's float-to-tick method (`jnp.round(jnp.float32(p) * jnp.float32(round(1.0 / tick)))`) and claims exactness for **on-tick** inputs only. Near $|p| \approx 9{,}999$, float32 resolution is coarse. Adjacent float32 values are $2^{-10}$ EUR/MWh $\approx 0.1$ tick apart, and $p \times 100 \approx 999{,}900$ has a float32 spacing of $0.0625$ tick. A method that works in float32 can therefore commit an off-grid price lying within about $0.08$ tick of a half-tick boundary to the other tick. The scenario's wording ("less than half a tick is snapped to the nearest tick") is exact, so this expectation is exact. Suppose draws fail **only** on the near-bound off-grid cases, and **only** with offsets within about $0.08$ tick of $\pm 0.5$. That is a float32-resolution finding, and it needs a domain ruling: is the snap exact, or is float32 resolution accepted (and if so, should the generator draw outside that band)? Section 7 prints the offsets of every failing case, so this pattern is visible at once.
3. *Which side, which book.* The generator's C01 cases carry no side and no book. The side (seed parity) and the background order are this scenario's choices. They make trading impossible and ensure every rejection meets a non-empty book.
4. *Hidden quantity.* "Hidden total quantity and visible peak quantity" (§4.2) is read as total = visible + hidden. On acceptance, (A) therefore asserts hidden $= n_q - n_k$.
5. *Rejection reason.* The engine does not expose which condition rejected an order. (T) asserts only the record builder's coarse reason `validator`, as opposed to `contract_not_open`.
6. *The PPD's units (r2).* `market-spec.md` says the PPD field carries the per-slice price increment, and §4.2's ICEwPI bullet prices each slice at the previous one "± a fixed delta", so the PPD is read as a price difference in EUR/MWh on §2.3's price grid, with no offset ($n_d = d / \tau_p$). The controls use one price tick, on the grid, so no snap question arises for the PPD.
7. *The sign rule is new (r2).* It was added to §5.4 in v2.14 (2026-10-07) and implemented by `OMIE-ICEWPI-SIGN-REJECT-1`; the r1 expectation could not assert it. The right-sign control is accepted under (A) like any one-slice iceberg; its first slice rests at the base price (§4.2: the increment applies from the next slice on).
8. *The slice margin is not a market rule (r2).* At most 64 slices per iceberg is the generator's margin under the engine's 64-slice ceiling (`scenario-ladder.md`, "The capacity-validity precondition"; `jax_constants.py` §B2), not anything in `market-spec.md`. C01 keeps every accepted iceberg inside it so that a capacity discard can never decide which draws are judged.

In [ ]:
NP = int((SPEC["price_max"] - SPEC["price_min"]) / SPEC["price_tick"])   # 1,999,800
NQMIN = int(SPEC["qty_min"] / SPEC["qty_tick"])                          # 1
NQMAX = int(SPEC["qty_max"] / SPEC["qty_tick"])                          # 9,990


def ppd_sign_ok(side, n_d):
    """§5.4 (v2.14): a BUY with PPD > 0 or a SELL with PPD < 0 is rejected; PPD = 0 always passes."""
    return n_d <= 0 if side == BUY else n_d >= 0


def expectation(case):
    """(V), and (A) on accept, for one case, from its own drawn values (never from draw.expected)."""
    n_p, _ = price_ticks(case["price_eur"])
    n_q, _ = qty_ticks(case["qty_mw"])
    ice = case["exec_restriction"] == ICE
    n_k = qty_ticks(case["peak_qty_mw"])[0] if ice else None
    n_d = ppd_ticks(case["ppd_eur"])[0] if ice else None
    ok = (n_p is not None and n_q is not None and 0 <= n_p <= NP and NQMIN <= n_q <= NQMAX
          and (not ice or (n_k is not None and n_d is not None and 0 < n_k <= n_q
                           and ppd_sign_ok(case["side"], n_d))))
    e = {"accept": bool(ok), "n_fills": 0}
    if ok:
        visible = n_k if ice else n_q
        e.update(price_int=n_p, visible_qty_int=visible, hidden_qty_int=n_q - visible,
                 slices=-(-n_q // n_k) if ice else None)
    return e


# The class table of the markdown above, as data: it must agree with what expectation() derives.
CLASS_TABLE = {
    "in_bounds_on_tick": (True, "p0"), "price_below_min": (False, -100), "price_above_max": (False, NP + 100),
    "qty_below_min": (False, "p0"), "qty_above_max": (False, "p0"),
    "sub_half_tick_off_grid": (True, "p0"), "over_half_tick_off_grid": (True, "p0+1"),
    "near_max_sub_half_tick": (True, NP), "near_max_over_half_tick": (False, NP + 1),
    "iceberg_peak_exceeds_qty": (False, "p0"), "iceberg_peak_zero": (False, "p0"),
    "price_at_min": (True, 0), "price_at_max": (True, NP),
    "price_one_tick_below_min": (False, -1), "price_one_tick_above_max": (False, NP + 1),
    "near_min_sub_half_tick": (True, 0), "near_min_over_half_tick": (False, -1),
    "qty_at_min": (True, "p0"), "qty_at_max": (True, "p0"),
    "qty_zero": (False, "p0"), "qty_one_tick_above_max": (False, "p0"),
    "iceberg_peak_equals_qty": (True, "p0"), "iceberg_peak_one_tick": (True, "p0"),
    "iceberg_peak_one_tick_over_qty": (False, "p0"), "iceberg_peak_negative": (False, "p0"),
    "icewpi_right_sign": (True, "p0"), "icewpi_wrong_sign": (False, "p0"),
}
# The class table's PPD column: the committed n_d per side (BUY, SELL) for the two ICEwPI controls.
PPD_TABLE = {"icewpi_right_sign": (-1, +1), "icewpi_wrong_sign": (+1, -1)}


def class_table_disagreements(seed):
    """Cases of `seed` whose derived (verdict, committed price, PPD) differ from the class table."""
    p0 = BACKGROUND[seed]["price_int"]
    bad = []
    names = [c["name"] for c in CASES[seed]]
    if sorted(names) != sorted(CLASS_TABLE) or len(names) != len(set(names)):
        bad.append("case set differs from the class table")
    for c in CASES[seed]:
        if c["name"] not in CLASS_TABLE:
            bad.append(c["name"])
            continue
        want_ok, want_p = CLASS_TABLE[c["name"]]
        want_p = p0 if want_p == "p0" else p0 + 1 if want_p == "p0+1" else want_p
        e = expectation(c)
        if e["accept"] != want_ok or c["price_int"] != want_p:
            bad.append(c["name"])
        elif c["name"] in PPD_TABLE and c["ppd_int"] != PPD_TABLE[c["name"]][0 if c["side"] == BUY else 1]:
            bad.append(c["name"])
        elif e["accept"] and e["slices"] is not None and not (1 <= e["slices"] <= SLICE_MARGIN):
            bad.append(c["name"])
    return bad


def max_slices(seed):
    """The largest ceil(n_q / n_k) over the seed's icebergs the expectation accepts (0 if none)."""
    return max([e["slices"] for e in EXPECTED[seed] if e["accept"] and e["slices"] is not None], default=0)


EXPECTED = {s: [expectation(c) for c in CASES[s]] for s in SEEDS}
n_acc = sum(e["accept"] for v in EXPECTED.values() for e in v)
n_all = sum(len(v) for v in EXPECTED.values())
print(f"{n_all} cases over {len(SEEDS)} seeds: {n_acc} accept, {n_all - n_acc} reject")
print("class table vs derived, seeds that disagree (full list):",
      [s for s in SEEDS if class_table_disagreements(s)] or "none")
_ms = {s: max_slices(s) for s in SEEDS}
print(f"slices per accepted iceberg: largest {max(_ms.values())} (margin {SLICE_MARGIN}); "
      f"seeds over the margin (full list): {[s for s in SEEDS if _ms[s] > SLICE_MARGIN] or 'none'}")
for c, e in zip(CASES[SEED0], EXPECTED[SEED0]):
    if e["accept"]:
        print(f"seed {SEED0} {c['name']:<32} accept: rests at price_int {e['price_int']} "
              f"({decode_price(e['price_int'], CONFIG):.2f} EUR/MWh), visible {e['visible_qty_int']} "
              f"({decode_qty(e['visible_qty_int'], CONFIG):.1f} MW), hidden {e['hidden_qty_int']}"
              + ("" if e["slices"] is None else f", {e['slices']} slices, ppd_int {c['ppd_int']:+d}"))
    else:
        print(f"seed {SEED0} {c['name']:<32} reject: book byte-identical, ts_counter unchanged")

## 6. Run

### Eager and jitted (wired at K3)

> **K3: this section is yours; sections 1–5 and 7 are locked (independence rule 2).** A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly these two names. Until both are callable, it records a failing `section 6 not wired` check and the notebook ends `NOT CERTIFIED`.

1. **`run_float_cases(seed, *, jit) -> dict`**. Submits the seed's background order and then its 27 cases (`BACKGROUND[seed]`, then `CASES[seed]`, with the order ids, sides and execution restrictions given there), one by one, to **one fresh book** on an OPEN contract. The submissions go through the production entry point that takes prices and quantities in **domain units** (EUR/MWh, MW, peak in MW, PPD in EUR/MWh). Each case's price, quantity, peak and PPD are passed **as drawn** (`case["price_eur"]`, `case["qty_mw"]`, `case["peak_qty_mw"]`, `case["ppd_eur"]`), **unsnapped**. Rounding them in the notebook first would take the snap out of the engine and make the off-grid rows vacuous. `jit=False` runs under `jax.disable_jit()`; `jit=True` runs jitted. The returned dict has exactly these keys:
   - `"path"`: `str`, the production entry point exercised (recorded as evidence).
   - `"background"`: `{"accepted": bool, "post": BookSnapshot}`, the background order's engine report and the book after it.
   - `"cases"`: a list in `CASES[seed]` order, one dict per case, with these keys:
     - `"name"` (`str`, the case's name) and `"oid"` (`int`);
     - `"submitted_price_eur"`, `"submitted_qty_mw"` (`float`) `"submitted_peak_mw"` and `"submitted_ppd_eur"` (`float`, or `None` for NON): the values actually passed to the entry point;
     - `"accepted"` (`bool`): the engine's own report;
     - `"pre"` and `"post"` (`tests.omiecid.reference.omie_invariants.BookSnapshot`): the book just before the submission and just after it returns. Section 7 reads their leaves `bids`, `asks`, `trades`, `iceberg_meta`, `n_active`, `ts_counter`, `book_full` and `truncated`, and their `resting`, `visible_qty` and `hidden_qty` readers;
     - `"n_fills"` (`int`): the trade rows that submission wrote.
2. **`run_tape(seed, *, jit) -> RunRecord`**: the record builder on the seed's integer tape, `tests.omiecid.reference.omie_invariants.build_record(TAPES[seed], jit=jit)`, on an OPEN contract with the default configs.

Section 7 calls both with `jit=True` on every seed, and with `jit=False` on `EAGER_SEEDS`. A seed beyond `DRAWS` (a re-draw after an I8 discard) is prepared with `prepare(seed)` before the call. If no production entry point accepts unsnapped domain-unit prices without editing locked or forbidden code, that is a `SCOPE-CHANGE` candidate, not a licence to pre-snap.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_float_cases and run_tape, per the contract in the markdown above.
#
# Float path: ContractBook.submit_order, the production entry point that takes an OMIEOrder
# in domain units (price EUR/MWh, quantity and peak MW, PPD EUR/MWh). It enters the module-level jitted
# _submit_order body, where the validator itself encodes the float price and quantity to
# integer ticks. The drawn floats (price, quantity, peak, PPD) are passed as they are:
# nothing is snapped here.
# Timestamps follow the integer tape's convention (TapeBuilder: 1, 2, ... in submission
# order, which here equals the order id), so both paths submit the same orders.
# Integer path: the battery's record builder on TAPES[seed].
# ---------------------------------------------------------------------------------------
import contextlib

import jax

from omiecid_simulator.jaxob.jaxob_constants import TradesFeat
from omiecid_simulator.omiecid.contract import Contract, ContractStatus
from omiecid_simulator.omiecid.contract_book import ContractBook
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig
from omiecid_simulator.omiecid.order_types import OMIEOrder, Status, Validity
from tests.omiecid.reference.omie_invariants import BookSnapshot, build_record

FLOAT_PATH = "omiecid_simulator.omiecid.contract_book.ContractBook.submit_order (OMIEOrder in EUR/MWh, MW)"
MARKET = OMIEMarketConfig()
# The record builder's contract: id 0, delivery 09:15-09:30, GCT 08:15 (minutes from midnight), OPEN.
_CONTRACT_ID, _DELIVERY_START, _DELIVERY_END, _GCT = 0, 555, 570, 495
_TR_SEC = TradesFeat.SEC.value   # a written trade row has SEC != -1


def _new_book():
    """One fresh book on one OPEN contract, with the default configs."""
    contract = Contract(contract_id=_CONTRACT_ID, delivery_start_minutes=_DELIVERY_START,
                        delivery_end_minutes=_DELIVERY_END, gct_minutes=_GCT,
                        status=ContractStatus.OPEN)
    book = ContractBook(contract, CONFIG, MARKET)
    return book, book.init()


def _snap(state):
    """A BookSnapshot of the engine state, leaf by leaf (copies, as the record builder takes them)."""
    n = int(state.n_active)
    return BookSnapshot(
        bids=np.array(state.lob.bids, copy=True), asks=np.array(state.lob.asks, copy=True),
        trades=np.array(state.lob.trades, copy=True),
        iceberg_meta=np.array(np.asarray(state.iceberg_meta)[:n], copy=True),
        n_active=n, ts_counter=int(state.ts_counter),
        book_full=bool(state.book_full), truncated=bool(state.truncated))


def _trade_rows_written(pre, post):
    """Trade rows one submission wrote: rows that changed between pre and post and are in use."""
    a, b = np.asarray(pre.trades), np.asarray(post.trades)
    if a.shape != b.shape:
        return -1
    changed = np.nonzero(np.any(a != b, axis=1))[0]
    return int(sum(1 for r in changed if int(b[r, _TR_SEC]) != -1))


def _order(o):
    """The OMIEOrder for one row of section 4, in domain units, with the drawn floats as they are.

    An iceberg carries its PPD (0 for a standard iceberg); a NON order carries none (ppd=0.0).
    """
    ice = o["exec_restriction"] == ICE
    return OMIEOrder(
        contract_id=_CONTRACT_ID, side=int(o["side"]), price=float(o["price_eur"]),
        quantity=float(o["qty_mw"]), order_id=int(o["oid"]), agent_id=1, timestamp=int(o["oid"]),
        exec_restriction=int(o["exec_restriction"]), validity=int(Validity.GFS), expiry_time=0,
        status=int(Status.ACTIVE), peak_qty=float(o["peak_qty_mw"]) if ice else 0.0,
        ppd=float(o["ppd_eur"]) if ice else 0.0)


def run_float_cases(seed, *, jit):
    """The background order, then the seed's 27 cases, unsnapped, on one fresh OPEN book."""
    with contextlib.nullcontext() if jit else jax.disable_jit():
        book, state = _new_book()
        state, res = book.submit_order(state, _order(BACKGROUND[seed]))
        out = {"path": FLOAT_PATH,
               "background": {"accepted": bool(res.accepted), "post": _snap(state)},
               "cases": []}
        for c in CASES[seed]:
            order = _order(c)
            pre = _snap(state)
            state, res = book.submit_order(state, order)
            post = _snap(state)
            out["cases"].append({
                "name": c["name"], "oid": int(c["oid"]),
                "submitted_price_eur": order.price, "submitted_qty_mw": order.quantity,
                "submitted_peak_mw": order.peak_qty if c["exec_restriction"] == ICE else None,
                "submitted_ppd_eur": order.ppd if c["exec_restriction"] == ICE else None,
                "accepted": bool(res.accepted), "pre": pre, "post": post,
                "n_fills": _trade_rows_written(pre, post),
            })
    return out


def run_tape(seed, *, jit):
    """The record builder on the seed's integer tape (OPEN contract, default configs)."""
    return build_record(TAPES[seed], jit=jit)


print(f"float path: {FLOAT_PATH}")

## 7. Certification

The oracle comparison in integer ticks, order ids and booleans comes first. Then the invariant battery (RB, I1–I9) runs on each seed's integer-tape record and is shown as a pass/fail table, with the capacity peaks, the largest slice count and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**How the assertions are recorded (r2).** With 500 draws of 27 cases each, one row per case per seed would bury the table. Each assertion is therefore one row per **case and criterion**, over every valid draw, and it passes only if every seed passes. Every row carries its **per-draw values in full**, pass or fail:
- `expected` and `observed` are lists of `[seed, value]` pairs, one per valid seed, in run order: the committed integers (`price_int`, visible and hidden `qty_int`), the verdicts and the state diffs. Nothing is `null` on a pass, and no list is cut to its first *k* entries.
- `seed` is the first failing seed, or `null` when no seed fails.
- `detail` gives the number of failing seeds and **all** of them; for off-grid cases, also **all** their offsets in ticks (section 5, note 2).

**Eager on the first valid seeds (r2).** A seed runs eager only after its jitted integer-tape run has passed I8, so a discarded seed is never the eager evidence. The eager seeds are chosen as section 3 says (one per side), recorded in `EAGER_SEEDS`, and a separate assertion checks that `EAGER_DRAWS` valid seeds ran eager and that both sides are covered.

**The battery.** It is folded per seed as usual, with `expects_trade=False`: the scenario is built so that nothing trades. I9's non-vacuity is carried by the accepted background order and the accepted cases. A check reported `not_applicable` on a draw is **vacuously satisfied** on that draw: it is never counted as a failure, and nothing here tests a battery status with `!= "pass"`.

In [ ]:
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Order validation and rejection",
    known_result="accept iff 0 <= n_p <= 1,999,800 and 1 <= n_q <= 9,990 and (ICE => 0 < n_k <= n_q and "
                 "PPD sign valid: BUY n_d <= 0, SELL n_d >= 0), n = nint after the tick snap; on accept the "
                 "order rests at n_p with visible n_q (NON) or n_k (ICE); on reject a byte-identical book",
    oracle_kind="exact-parametric (M4 no-mutation relation)", quotes=QUOTES, citations=CITATIONS,
    constructions=(CONSTRUCTION,), requested_draws=DRAWS, runtime_forecast_s=180.0, started_at=T0,
    downstream="every later scenario that draws random orders (C02-C17) and invariant I9: a silently "
               "rejecting validator would let them pass vacuously; C08-C10 rely on the iceberg peak constraint, "
               "C09 on the ICEwPI sign rule",
)

LEAVES = ("bids", "asks", "trades", "iceberg_meta", "n_active", "ts_counter", "book_full", "truncated")
R_BOUNDS, R_PEAK = "§2.3, §5.4", "§4.2 (derived; review §10); §5.4 (v2.14, peak > total)"
R_REJECT, R_SNAP = "§5.1, §5.2 (M4 no-mutation)", "§2.3; review §2; OQ-16"
R_PPD = "§5.4 (v2.14, ICEwPI sign); §4.2"


def leaf_diff(a, b):
    """Names of the state leaves on which a and b differ (arrays: shape, dtype and bytes)."""
    out = []
    for k in LEAVES:
        x, y = getattr(a, k, None), getattr(b, k, None)
        if isinstance(x, np.ndarray) or isinstance(y, np.ndarray):
            x, y = np.asarray(x), np.asarray(y)
            same = x.shape == y.shape and x.dtype == y.dtype and np.array_equal(x, y)
        else:
            same = x == y
        if not same:
            out.append(k)
    return out


def rule_of(name):
    if name.startswith("icewpi"):
        return R_PPD
    if name.startswith("iceberg"):
        return R_PEAK
    if "half_tick" in name:
        return R_SNAP
    return R_BOUNDS


def accept_delta(c, e, pre, post):
    """(A): what is wrong with post against pre for an accepted case (empty list = as expected)."""
    side, other, oid = c["side"], 1 - c["side"], c["oid"]
    bad = []
    new = [r for r in post.resting(side) if r[2] == oid]
    if len(new) != 1:
        bad.append(f"{len(new)} resting entries with oid {oid} (expected 1)")
    else:
        if new[0][0] != e["price_int"]:
            bad.append(f"rests at price_int {new[0][0]} (expected {e['price_int']})")
        if new[0][1] != e["visible_qty_int"]:
            bad.append(f"visible qty_int {new[0][1]} (expected {e['visible_qty_int']})")
    if any(r[2] == oid for r in pre.resting(side)):
        bad.append(f"oid {oid} was already resting before its submission")
    if sorted(r for r in post.resting(side) if r[2] != oid) != sorted(pre.resting(side)):
        bad.append("other resting entries on the side changed")
    if sorted(post.resting(other)) != sorted(pre.resting(other)):
        bad.append("the other side changed")
    if not np.array_equal(np.asarray(post.trades), np.asarray(pre.trades)):
        bad.append("the trade tape changed")
    if bool(post.book_full) or bool(post.truncated):
        bad.append("a capacity flag was raised")
    return bad


# ---- accumulation: per (case, criterion), every valid seed, in full ----------------------
RECORD = defaultdict(list)   # (case, criterion) -> [(seed, expected, observed, offset_ticks, ok)], all seeds
UNITS = {"verdict (float path)": "bool", "verdict (integer tape)": "other", "entered price_int": "price_int",
         "visible qty_int": "qty_int", "hidden qty_int": "qty_int", "book change on accept": "other",
         "book byte-identical on reject": "other", "no trade written": "count", "submitted unsnapped": "other",
         "accepted and resting": "bool", "accepted on the integer tape": "bool",
         "every case reported, in order": "count", "no trade on the integer tape": "count"}


def note(case, criterion, seed, ok, expected, observed):
    """Record one seed's value for (case, criterion), pass or fail: per-draw values are always kept."""
    name = case["name"] if isinstance(case, dict) else str(case)
    off = case.get("offset_ticks") if isinstance(case, dict) else None
    RECORD[(name, criterion)].append((seed, expected, observed, off, bool(ok)))


def guarded(label, seed, fn, *args, **kw):
    """Call a section-6 function; a raise is recorded as a failed check, never propagated."""
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, detail=f"{type(exc).__name__}: {exc} | "
                   + traceback.format_exc().splitlines()[-1], rule=R_BOUNDS)
        return None


def check_float_run(seed, fr):
    """(V), (A) and (R) on one seed's float-path run."""
    bg = fr.get("background") or {}
    try:
        bg_ok = bool(bg.get("accepted")) and any(r[2] == 1 for r in bg["post"].resting(SIDES[seed]))
    except Exception:  # noqa: BLE001
        bg_ok = False
    note("background", "accepted and resting", seed, bg_ok, True, bg_ok)
    got = list(fr.get("cases") or [])
    names_ok = [g.get("name") for g in got] == [c["name"] for c in CASES[seed]]
    note("run", "every case reported, in order", seed, names_ok, len(CASES[seed]), len(got))
    if not names_ok:
        return
    for c, e, g in zip(CASES[seed], EXPECTED[seed], got):
        sub = (g.get("submitted_price_eur"), g.get("submitted_qty_mw"), g.get("submitted_peak_mw"),
               g.get("submitted_ppd_eur"))
        want = (c["price_eur"], c["qty_mw"], c["peak_qty_mw"], c["ppd_eur"])
        note(c, "submitted unsnapped", seed, sub == want and g.get("oid") == c["oid"],
             [c["oid"], *want], [g.get("oid"), *sub])
        acc = bool(g.get("accepted"))
        note(c, "verdict (float path)", seed, acc == e["accept"], e["accept"], acc)
        n_f = g.get("n_fills")
        note(c, "no trade written", seed, n_f == 0, 0, n_f)
        pre, post = g.get("pre"), g.get("post")
        if pre is None or post is None:
            note(c, "book change on accept" if e["accept"] else "book byte-identical on reject", seed,
                 False, "pre and post snapshots", None)
            continue
        if e["accept"]:
            bad = accept_delta(c, e, pre, post)
            note(c, "book change on accept", seed, bad == [], [], bad)
            rest = [r for r in post.resting(c["side"]) if r[2] == c["oid"]]
            p_obs = int(rest[0][0]) if len(rest) == 1 else None
            note(c, "entered price_int", seed, p_obs == e["price_int"], e["price_int"], p_obs)
            vis = post.visible_qty(c["oid"], c["side"])
            vis = None if vis is None else int(vis)
            note(c, "visible qty_int", seed, vis == e["visible_qty_int"], e["visible_qty_int"], vis)
            hid = post.hidden_qty(c["oid"])
            hid = None if hid is None else int(hid)
            note(c, "hidden qty_int", seed, hid == e["hidden_qty_int"], e["hidden_qty_int"], hid)
        else:
            d = leaf_diff(pre, post)
            note(c, "book byte-identical on reject", seed, d == [], [], d)


def check_tape_record(seed, rec):
    """(T) on one seed's integer-tape record."""
    rej = {r.oid: r.reason for r in rec.rejections}
    note("background", "accepted on the integer tape", seed, 1 in rec.accepted, True, 1 in rec.accepted)
    for c, e in zip(CASES[seed], EXPECTED[seed]):
        obs = "accepted" if c["oid"] in rec.accepted else f"rejected ({rej.get(c['oid'])})"
        want = "accepted" if e["accept"] else "rejected (validator)"
        note(c, "verdict (integer tape)", seed, obs == want, want, obs)
    note("run", "no trade on the integer tape", seed, len(rec.fills) == 0, 0, len(rec.fills))


def _snap_key(s):
    if s is None:
        return None
    out = []
    for k in LEAVES:
        v = getattr(s, k, None)
        out.append((np.asarray(v).tobytes(), np.asarray(v).shape, str(np.asarray(v).dtype))
                   if isinstance(v, np.ndarray) else v)
    return tuple(out)


def float_run_key(fr):
    """The integer content of a float-path run, for the eager/jitted comparison."""
    bg = fr.get("background") or {}
    return (bool(bg.get("accepted")), _snap_key(bg.get("post")),
            tuple((g.get("name"), bool(g.get("accepted")), g.get("n_fills"), _snap_key(g.get("pre")),
                   _snap_key(g.get("post"))) for g in (fr.get("cases") or [])))


def wants_eager(seed):
    """Eager on the first valid seeds: while fewer than EAGER_DRAWS ran, and one per side first."""
    if len(EAGER_SEEDS) >= EAGER_DRAWS:
        return False
    covered = {SIDES[s] for s in EAGER_SEEDS}
    return SIDES[seed] not in covered or covered >= {BUY, SELL}


# ---- the configuration is §2.3's ----------------------------------------------------------
cfg = {k: Fraction(str(getattr(CONFIG, k))) for k in SPEC}
cert.check("OMIEProductConfig equals §2.3 (bounds and ticks)", cfg == SPEC,
           expected={k: str(v) for k, v in SPEC.items()}, observed={k: str(v) for k, v in cfg.items()},
           unit="other", rule="§2.3")

wired = callable(globals().get("run_float_cases")) and callable(globals().get("run_tape"))
cert.check("section 6 wired (run_float_cases, run_tape defined)", wired, expected=True, observed=wired,
           unit="bool", detail="" if wired else "section 6 is still the K2 stub", rule=R_BOUNDS)

PATHS = set()
VALID_SEEDS = []    # seeds judged, in run order (I8 valid, or the tape run raised and the draw is still judged)
RUN_SEEDS = list(SEEDS)
if wired:
    RUN_SEEDS = []
    valid, seed = 0, SEED0
    while valid < DRAWS and seed < SEED0 + MAX_ATTEMPTS:
        if seed not in CASES:
            prepare(seed)
        if seed not in EXPECTED:
            EXPECTED[seed] = [expectation(c) for c in CASES[seed]]
        RUN_SEEDS.append(seed)
        rec = guarded(f"seed {seed}: integer tape (jit)", seed, run_tape, seed, jit=True)
        if rec is not None and not cert.add_battery(run_battery(rec, expects_trade=False), seed=seed):
            seed += 1   # I8 invalid: discarded, re-drawn with the next seed; never run eager
            continue
        valid += 1
        VALID_SEEDS.append(seed)
        cert.record_case(seed, TAPES[seed], {"construction_params": DRAWN[seed].params,
                                             "side": SIDES[seed], "cases": CASES[seed]})
        if rec is not None:
            cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
            check_tape_record(seed, rec)
        fr = guarded(f"seed {seed}: float cases (jit)", seed, run_float_cases, seed, jit=True)
        if isinstance(fr, dict):
            PATHS.add(str(fr.get("path")))
            check_float_run(seed, fr)
        elif fr is not None:
            cert.check(f"seed {seed}: run_float_cases returned a dict", False, seed=seed,
                       observed=type(fr).__name__, rule=R_BOUNDS)
        if rec is not None and wants_eager(seed):   # a seed confirmed valid by I8, chosen as it comes
            EAGER_SEEDS.append(seed)
            rec_e = guarded(f"seed {seed}: integer tape (eager)", seed, run_tape, seed, jit=False)
            fr_e = guarded(f"seed {seed}: float cases (eager)", seed, run_float_cases, seed, jit=False)
            diffs = []
            if rec_e is not None:
                diffs += [f"tape.{d}" for d in compare_records(rec_e, rec)]
            else:
                diffs.append("integer-tape run missing")
            if isinstance(fr, dict) and isinstance(fr_e, dict):
                if float_run_key(fr_e) != float_run_key(fr):
                    diffs.append("float-path run differs (accepted / n_fills / pre / post)")
            else:
                diffs.append("float-path run missing")
            cert.set_eager_jit(seed, diffs == [], detail=", ".join(diffs))
        seed += 1

    # ---- eager ran on the first valid seeds, one per side -----------------------------------
    want_n = min(EAGER_DRAWS, len(VALID_SEEDS))
    want_sides = {SIDES[s] for s in VALID_SEEDS} if EAGER_DRAWS >= 2 else set()
    got_sides = {SIDES[s] for s in EAGER_SEEDS}
    eager_ok = (len(EAGER_SEEDS) == want_n and set(EAGER_SEEDS) <= set(VALID_SEEDS)
                and want_sides <= got_sides)
    cert.check("eager runs on the first valid seeds (one per side)", eager_ok,
               expected={"n": want_n, "sides": sorted(want_sides), "subset_of_valid": True},
               observed={"n": len(EAGER_SEEDS), "seeds": list(EAGER_SEEDS), "sides": sorted(got_sides),
                         "subset_of_valid": set(EAGER_SEEDS) <= set(VALID_SEEDS)},
               unit="other", detail=f"eager seeds {list(EAGER_SEEDS)}; valid seeds {len(VALID_SEEDS)}",
               rule=R_BOUNDS)

# ---- the dataset is what section 5 assumes (every seed run, full lists) ---------------------
bad_tbl = [s for s in RUN_SEEDS if class_table_disagreements(s)]
cert.check("dataset: every seed's cases fall in their class (section 5 table)", bad_tbl == [],
           expected=[], observed=bad_tbl, unit="other",
           detail=(f"{len(bad_tbl)}/{len(RUN_SEEDS)} seeds disagree: "
                   f"{ {s: class_table_disagreements(s) for s in bad_tbl} }") if bad_tbl
           else f"{len(RUN_SEEDS)}/{len(RUN_SEEDS)} seeds", rule=R_SNAP)
_gaps = [[s, min(c["tie_gap"] for c in CASES[s])] for s in RUN_SEEDS]
_gap_min = min((g for _, g in _gaps), default=TIE_GAP_MIN)
cert.check("dataset: no case at a half-tick tie", _gap_min > 0, expected=True, observed=_gap_min > 0,
           unit="bool", detail=f"smallest distance {_gap_min:.4f} tick over {len(RUN_SEEDS)} seeds", rule=R_SNAP)
_slices = [[s, max_slices(s)] for s in RUN_SEEDS]
_over = [s for s, n in _slices if n > SLICE_MARGIN]
cert.check(f"dataset: every accepted iceberg within the slice margin (ceil(n_q/n_k) <= {SLICE_MARGIN}, pre-run)",
           _over == [], expected=[[s, f"<= {SLICE_MARGIN}"] for s in RUN_SEEDS], observed=_slices,
           unit="count", detail=f"largest {max((n for _, n in _slices), default=0)}; "
                                f"{len(_over)} seeds over the margin: {_over}", rule=R_PEAK)
_one = [[s, next(c["qty_int"] for c in CASES[s] if c["name"] == "iceberg_peak_one_tick")] for s in RUN_SEEDS]
print(f"iceberg_peak_one_tick slice counts over {len(_one)} seeds: min {min(n for _, n in _one)}, "
      f"max {max(n for _, n in _one)}, distinct {len({n for _, n in _one})}")

if wired:
    # ---- fold the per-(case, criterion) record into assertions: full per-draw lists -------
    for key in sorted(RECORD):
        name, crit = key
        recs = RECORD[key]
        fails = [r for r in recs if not r[4]]
        det = f"{len(recs) - len(fails)}/{len(recs)} seeds pass"
        if fails:
            det = f"{len(fails)}/{len(recs)} seeds fail; failing seeds: {[f[0] for f in fails]}"
            if any(f[3] is not None for f in fails):
                det += f"; their offsets (ticks): {[None if f[3] is None else round(f[3], 4) for f in fails]}"
        cert.check(f"{name}: {crit}", not fails, seed=fails[0][0] if fails else None,
                   expected=[[r[0], r[1]] for r in recs], observed=[[r[0], r[2]] for r in recs],
                   unit=UNITS.get(crit, "other"), detail=det,
                   rule=R_REJECT if "byte-identical" in crit else rule_of(name))
    cert.check("float path named (evidence)", len(PATHS) == 1 and "None" not in PATHS,
               expected=1, observed=sorted(PATHS), unit="other", rule=R_BOUNDS)

display(Markdown(cert.table()))
print(f"float path: {sorted(PATHS) or '(not run)'}; eager seeds: {EAGER_SEEDS}; "
      f"valid seeds: {len(VALID_SEEDS)} of {len(RUN_SEEDS)} tried")

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")


In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))
